# Put–Call Parity Arbitrage (European Options)

This notebook is a **finance-first** mini project.

You enter market inputs (**S, K, T, r, Call, Put**) and the notebook:

- checks **Put–Call Parity**  
- detects a **theoretical arbitrage** if parity is violated (beyond a tolerance)  
- prints the **exact trade** (buy/sell call/put/stock + borrow/lend PV(K))  

> Educational project: real trading needs **bid/ask**, fees, margin, and shorting constraints.


## 1) Theory (no dividends)

For European options (continuous compounding):

\[
C - P = S - K e^{-rT}
\]

Where:

- **S**: spot price  
- **K**: strike  
- **T**: time to maturity (in years)  
- **r**: risk-free rate (continuous, decimal)  
- **C, P**: call and put prices  

If the market violates the equality, there is a **riskless (theoretical) arbitrage**.


## 2) Setup

Run the cell below once. It defines simple functions and a helper that prints results nicely.


In [ ]:
import math

def pv_strike(K, r, T):
    """Present value of the strike: PV(K) = K * exp(-rT)."""
    return K * math.exp(-r * T)

def put_call_parity_arbitrage(S, K, T, r, C, P, tol=1e-4):
    """
    Put–Call Parity check for European options (no dividends):

        C - P = S - K*exp(-rT)

    Returns a dict with:
    - parity gap
    - arbitrage (True/False)
    - trade list if violated
    """
    PVK = pv_strike(K, r, T)
    lhs = C - P
    rhs = S - PVK
    gap = lhs - rhs

    res = {
        "PV(K)": PVK,
        "LHS (C-P)": lhs,
        "RHS (S-PV(K))": rhs,
        "Gap (LHS-RHS)": gap,
        "Arbitrage?": False,
        "Message": "No arbitrage: parity holds within tolerance.",
        "Trades": [],
        "Initial cashflow (t=0)": 0.0,
        "Maturity note": ""
    }

    if abs(gap) <= tol:
        return res

    res["Arbitrage?"] = True
    res["Maturity note"] = (
        "At maturity, the portfolio payoff nets to ~0 for any S_T; "
        "profit is the positive initial cashflow (ignoring costs)."
    )

    if gap > tol:
        res["Message"] = "(C - P) is TOO EXPENSIVE → Sell (C-P), buy (S - PV(K))."
        res["Trades"] = [
            "SELL 1 Call  (+C)",
            "BUY  1 Put   (-P)",
            "BUY  1 Stock (-S)",
            f"BORROW PV(K) (+{PVK:.6f}) → repay K at maturity"
        ]
        res["Initial cashflow (t=0)"] = C - P - S + PVK
    else:
        res["Message"] = "(C - P) is TOO CHEAP → Buy (C-P), sell (S - PV(K))."
        res["Trades"] = [
            "BUY  1 Call  (-C)",
            "SELL 1 Put   (+P)",
            "SHORT 1 Stock (+S)",
            f"LEND PV(K)   (-{PVK:.6f}) → receive K at maturity"
        ]
        res["Initial cashflow (t=0)"] = -C + P + S - PVK

    return res

def pretty_print(res):
    print("—" * 60)
    print("Put–Call Parity Check")
    print("—" * 60)
    print(f"PV(K)                 : {res['PV(K)']:.6f}")
    print(f"LHS (C - P)           : {res['LHS (C-P)']:.6f}")
    print(f"RHS (S - PV(K))       : {res['RHS (S-PV(K))']:.6f}")
    print(f"Gap (LHS - RHS)       : {res['Gap (LHS-RHS)']:.6f}")
    print(f"Arbitrage?            : {res['Arbitrage?']}")
    print(f"Message               : {res['Message']}")
    if res["Trades"]:
        print("\nTrades (1 unit):")
        for t in res["Trades"]:
            print("  -", t)
        print(f"\nInitial cashflow t=0 : {res['Initial cashflow (t=0)']:.6f}")
        print("Note                  :", res["Maturity note"])
    print("—" * 60)


## 3) Quick example

Try the example below, then change **P** (the put price) to see when an arbitrage appears.

Tip: in real markets you usually set `tol` to something like **0.01** (bid/ask noise).


In [ ]:
# Example (edit these numbers)
S = 31
K = 30
T = 0.25     # 3 months
r = 0.10     # 10% (continuous)
C = 3.00
P = 2.25     # try 2.25, then try 1.00

tol = 0.01

res = put_call_parity_arbitrage(S, K, T, r, C, P, tol=tol)
pretty_print(res)


## 4) Your turn (fill your own inputs)

Run this cell, enter your numbers, and the notebook prints the result.


In [ ]:
S = float(input("Spot S = "))
K = float(input("Strike K = "))
T = float(input("Maturity T (years) = "))
r = float(input("Risk-free r (continuous, decimal) = "))
C = float(input("Call price C = "))
P = float(input("Put price  P = "))
tol = float(input("Tolerance (e.g. 0.01) = "))

res = put_call_parity_arbitrage(S, K, T, r, C, P, tol=tol)
pretty_print(res)


## 5) Notes (what to mention in interviews)

- Parity is an **arbitrage relationship** (European options).  
- Real-life trading requires **bid/ask**, fees, margin, and the ability to short the stock.  
- Dividends change the parity (this notebook assumes **no dividends**).  
